# Portfolio Risk Analysis: Volatility, Correlation and Value at Risk

**Author:** Qais Sawar · BSc Financial Mathematics, University of Nottingham

This project measures how risky a 5-stock portfolio is, using the same core tools that bank and asset-manager risk teams use every day:

1. **Volatility** – how much each stock's price swings
2. **Correlation** – whether the stocks tend to move together (which decides how much diversification helps)
3. **Value at Risk (VaR)** – the most the portfolio is likely to lose in one day, at 95% confidence
4. **Backtesting** – checking whether the VaR model would actually have worked in the past
5. **Expected Shortfall** – the average loss on the worst days, beyond the VaR cut-off

**How to run it:** open this file in Google Colab, then click *Runtime → Run all*. Read each explanation before its code.

## Step 1: Set up

`yfinance` downloads free historical share prices from Yahoo Finance. Colab doesn't include it, so the first line installs it.

In [ ]:
!pip install yfinance -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy.stats import norm

## Step 2: Choose the portfolio

You can change any of these and re-run everything:

- **tickers** – the stocks in the portfolio (US ticker symbols)
- **weights** – how much of the money goes into each stock (must add up to 1)
- **portfolio_value** – how much the portfolio is worth, in £
- **confidence** – 0.95 means "95% confident"

I've chosen a mix of tech stocks (which tend to move together) and a bank, so the correlation step has something interesting to show.

In [ ]:
tickers = ["NVDA", "META", "AAPL", "MSFT", "JPM"]
weights = np.array([0.2, 0.2, 0.2, 0.2, 0.2])   # equal weights: 20% each
portfolio_value = 10_000                         # £10,000 portfolio
confidence = 0.95
start_date = "2021-01-01"

assert abs(weights.sum() - 1) < 1e-9, "Weights must add up to 1"

## Step 3: Download the price data

We download daily closing prices (adjusted for dividends and stock splits, so returns are accurate) and drop any days with missing data.

In [ ]:
data = yf.download(tickers, start=start_date, auto_adjust=True, progress=False)
prices = data["Close"][tickers].dropna()

print(f"Downloaded {len(prices)} trading days, from {prices.index[0].date()} to {prices.index[-1].date()}")
prices.tail()

To compare stocks with very different prices, we rebase each one to start at 100. A line at 250 means that stock is up 150% since the start.

In [ ]:
(prices / prices.iloc[0] * 100).plot(figsize=(10, 5), title="Growth of 100 invested in each stock")
plt.ylabel("Value (started at 100)")
plt.grid(alpha=0.3)
plt.show()

## Step 4: Daily returns

Risk is measured on **returns** (percentage changes), not prices. A daily return of 0.02 means the stock rose 2% that day.

$$r_t = \frac{P_t - P_{t-1}}{P_{t-1}}$$

In [ ]:
returns = prices.pct_change().dropna()
returns.head()

## Step 5: Volatility

Volatility is the **standard deviation** of returns: how spread out the daily moves are. A higher number means bigger swings, so more risk.

There are about 252 trading days in a year, and because variance adds up over independent days, we scale daily volatility by $\sqrt{252}$ to get **annual volatility**.

In [ ]:
daily_vol = returns.std()
annual_vol = daily_vol * np.sqrt(252)

vol_table = pd.DataFrame({"Daily volatility": daily_vol, "Annual volatility": annual_vol})
display(vol_table.style.format("{:.2%}"))

sorted_vol = annual_vol.sort_values()
plt.figure(figsize=(8, 4))
plt.barh(list(sorted_vol.index), sorted_vol.values)
plt.title("Annual volatility by stock")
plt.xlabel("Annual volatility")
plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
plt.grid(alpha=0.3, axis="x")
plt.show()

## Step 6: Correlation

Correlation runs from -1 to +1:

- **+1**: the two stocks always move the same way (no diversification benefit)
- **0**: no relationship
- **-1**: they always move in opposite directions (perfect hedge)

Stocks with high correlation tend to fall at the same time, which is exactly what a risk manager worries about.

In [ ]:
corr = returns.corr()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(tickers)))
ax.set_xticklabels(tickers)
ax.set_yticks(range(len(tickers)))
ax.set_yticklabels(tickers)
for i in range(len(tickers)):
    for j in range(len(tickers)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center",
                color="white" if corr.iloc[i, j] > 0.6 else "black")
fig.colorbar(im)
ax.set_title("Correlation of daily returns")
plt.show()

## Step 7: Portfolio returns and volatility

The portfolio's daily return is the weighted average of the stocks' returns.

Notice that the portfolio's volatility is **lower than the average of the individual volatilities**. That gap is the benefit of diversification, and it only exists because the stocks aren't perfectly correlated.

In [ ]:
portfolio_returns = returns @ weights

port_annual_vol = portfolio_returns.std() * np.sqrt(252)
avg_individual_vol = (annual_vol * weights).sum()

print(f"Weighted average of individual volatilities: {avg_individual_vol:.2%}")
print(f"Actual portfolio volatility:                 {port_annual_vol:.2%}")
print(f"Risk removed by diversification:             {avg_individual_vol - port_annual_vol:.2%}")

## Step 8: Value at Risk (VaR)

**95% one-day VaR** answers: *"On 95% of days, the portfolio should not lose more than £X."* Put the other way round, on roughly 1 day in 20, losses will be worse than £X.

We calculate it two ways:

1. **Historical VaR** – look at every past daily return, and take the cut-off for the worst 5% of days. No assumptions about the shape of returns.
2. **Parametric (normal) VaR** – assume returns follow a normal distribution, so VaR = 1.645 × standard deviation (minus the average return). Quick, but it underestimates extreme crashes, because real returns have "fat tails".

In [ ]:
# 1. Historical VaR: the 5th percentile of past daily returns
hist_var_pct = -np.percentile(portfolio_returns, (1 - confidence) * 100)

# 2. Parametric VaR: assumes returns are normally distributed
z = norm.ppf(confidence)                      # 1.645 for 95%
mu, sigma = portfolio_returns.mean(), portfolio_returns.std()
param_var_pct = -(mu - z * sigma)

print(f"95% one-day Historical VaR: {hist_var_pct:.2%}  =  £{hist_var_pct * portfolio_value:,.0f}")
print(f"95% one-day Parametric VaR: {param_var_pct:.2%}  =  £{param_var_pct * portfolio_value:,.0f}")

The histogram shows every daily return. The red line is the historical VaR cut-off: the bars to its left are the worst 5% of days.

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(portfolio_returns, bins=80, alpha=0.75)
plt.axvline(-hist_var_pct, color="red", linestyle="--", linewidth=2,
            label=f"95% Historical VaR ({-hist_var_pct:.2%})")
plt.title("Distribution of daily portfolio returns")
plt.xlabel("Daily return")
plt.ylabel("Number of days")
plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
plt.legend()
plt.grid(alpha=0.3)
plt.show()

worst_day = portfolio_returns.idxmin()
print(f"Worst single day: {worst_day.date()}, return {portfolio_returns.min():.2%} "
      f"(a loss of £{-portfolio_returns.min() * portfolio_value:,.0f})")

## Step 9: Backtesting – does the VaR model actually work?

A VaR model is only useful if it's right. At 95%, losses should be worse than VaR on about **5% of days**. Each such day is called a **breach**.

To test it fairly, we only use information that was available at the time: on each day, we calculate VaR from the **previous 250 trading days** (about one year), then check whether the next day's loss broke it.

- About 5% breaches → the model is working
- Far more than 5% → the model underestimates risk (dangerous)
- Far fewer than 5% → the model is too cautious

In [ ]:
window = 250
rolling_var = -portfolio_returns.rolling(window).quantile(1 - confidence).shift(1)

test = pd.DataFrame({"return": portfolio_returns, "VaR": rolling_var}).dropna()
test["breach"] = test["return"] < -test["VaR"]

breach_rate = test["breach"].mean()
print(f"Days tested: {len(test)}")
print(f"Breaches:    {test['breach'].sum()}  ({breach_rate:.1%} of days, expected {1 - confidence:.0%})")

plt.figure(figsize=(11, 5))
plt.plot(test.index, test["return"], linewidth=0.6, label="Daily return")
plt.plot(test.index, -test["VaR"], color="red", linewidth=1.2, label="95% VaR (rolling 250 days)")
breaches = test[test["breach"]]
plt.scatter(breaches.index, breaches["return"], color="red", s=12, zorder=3, label="Breach")
plt.title("VaR backtest: days when losses were worse than VaR")
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Step 10: Expected Shortfall

VaR gives the cut-off for the worst 5% of days, but it doesn't say how bad those days actually are. Expected Shortfall (ES) answers that: it's the average loss on the worst 5% of days.

Under the Basel rules for market risk, banks now use ES instead of VaR as their main measure, because it captures how big the crashes are, not just how often they happen.

- Historical ES: the average of the actual returns below the VaR cut-off
- Parametric ES: the normal formula $ES = -\left(\mu - \sigma \frac{\varphi(z)}{\alpha}\right)$, where $\varphi$ is the normal density and $\alpha$ = 5%

In [ ]:
alpha = 1 - confidence                         # 0.05

# 1. Historical ES: the average of the actual returns on the worst 5% of days
worst_days = portfolio_returns[portfolio_returns <= -hist_var_pct]
hist_es_pct = -worst_days.mean()

# 2. Parametric ES: the normal-distribution formula
param_es_pct = -(mu - sigma * norm.pdf(z) / alpha)

print(f"Worst 5% of days: {len(worst_days)} days out of {len(portfolio_returns)}")
print()
print(f"{'':28}{'VaR':>16}{'Expected Shortfall':>22}")
print(f"{'Historical':28}{hist_var_pct:>8.2%} = £{hist_var_pct * portfolio_value:>4,.0f}{hist_es_pct:>13.2%} = £{hist_es_pct * portfolio_value:,.0f}")
print(f"{'Parametric (normal)':28}{param_var_pct:>8.2%} = £{param_var_pct * portfolio_value:>4,.0f}{param_es_pct:>13.2%} = £{param_es_pct * portfolio_value:,.0f}")
print()
print(f"On the worst 5% of days, the average loss is {hist_es_pct / hist_var_pct:.2f}x the VaR figure.")

The red bars are the worst 5% of days. VaR is where they start, and ES is their average.

In [ ]:
plt.figure(figsize=(10, 5))
bins = np.linspace(portfolio_returns.min(), portfolio_returns.max(), 80)
plt.hist(portfolio_returns[portfolio_returns > -hist_var_pct], bins=bins, alpha=0.6, label="Normal days")
plt.hist(worst_days, bins=bins, color="red", alpha=0.8, label="Worst 5% of days")
plt.axvline(-hist_var_pct, color="black", linestyle="--", linewidth=2, label=f"95% VaR ({-hist_var_pct:.2%})")
plt.axvline(-hist_es_pct, color="darkred", linewidth=2, label=f"95% Expected Shortfall ({-hist_es_pct:.2%})")
plt.title("VaR is the cut-off, Expected Shortfall is the average loss beyond it")
plt.xlabel("Daily return")
plt.ylabel("Number of days")
plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
plt.legend()
plt.grid(alpha=0.3)
plt.show()

Basel uses ES at 97.5% confidence, which for normally distributed returns is about as strict as the old 99% VaR. If real returns have fat tails, the 97.5% ES comes out higher than the 99% VaR.

In [ ]:
basel_conf = 0.975
basel_var = -np.percentile(portfolio_returns, (1 - basel_conf) * 100)
basel_es = -portfolio_returns[portfolio_returns <= -basel_var].mean()

print(f"97.5% one-day Historical VaR: {basel_var:.2%}  =  £{basel_var * portfolio_value:,.0f}")
print(f"97.5% one-day Historical ES:  {basel_es:.2%}  =  £{basel_es * portfolio_value:,.0f}")
print()
print(f"97.5% ES compared with 99% Historical VaR: {basel_es:.2%} vs {-np.percentile(portfolio_returns, 1):.2%}")

## Step 11: Summary

In [ ]:
print("PORTFOLIO RISK SUMMARY")
print("=" * 50)
print(f"Stocks:                    {', '.join(tickers)}")
print(f"Period:                    {prices.index[0].date()} to {prices.index[-1].date()}")
print(f"Portfolio value:           £{portfolio_value:,.0f}")
print(f"Annual volatility:         {port_annual_vol:.2%}")
print(f"Diversification benefit:   {avg_individual_vol - port_annual_vol:.2%} of volatility removed")
print(f"95% 1-day Historical VaR:  £{hist_var_pct * portfolio_value:,.0f}")
print(f"95% 1-day Parametric VaR:  £{param_var_pct * portfolio_value:,.0f}")
print(f"Backtest breach rate:      {breach_rate:.1%} (expected {1 - confidence:.0%})")
print(f"95% 1-day Historical ES:   £{hist_es_pct * portfolio_value:,.0f}")
print(f"97.5% 1-day Historical ES: £{basel_es * portfolio_value:,.0f}")

## What I learned / limitations

- **VaR doesn't say how bad the worst days are.** It gives the cut-off for the worst 5%, not the size of losses beyond it. That's why I added Expected Shortfall in Step 10: the average loss on those worst days was clearly bigger than the VaR figure.
- **Historical VaR assumes the past repeats.** If the last few years were calm, it will underestimate a new crisis like 2008 or March 2020.
- **Normal VaR underestimates crashes**, because real returns have fatter tails than a normal distribution.
- **Correlations rise in a crisis.** Stocks that normally diversify each other tend to all fall together in a crash, exactly when diversification is needed most.

## My Results

My analysis found that the portfolio had an annual volatility of 25.78%. This is higher than the overall stock market (around 15–20%) because four of the five stocks are tech companies, which tend to be very volatile and sensitive to external factors such as conflicts or new laws affecting artificial intelligence.

However, diversification spreads the risk across multiple assets. If all five stocks always moved together, the volatility would have been 34.61%, so diversification reduced it by 8.83 percentage points, about a quarter of the risk, even though most of the portfolio is in one sector.

The historical VaR showed that on 95% of days, a £10,000 portfolio shouldn't lose more than £263. In other words, about 1 day in 20, roughly 12–13 trading days a year, will see a loss worse than £263.

The parametric VaR was lower, at £256. This is because it assumes daily returns follow a perfect bell curve (the normal distribution), which real markets don't, as extreme losses happen more often than a bell curve predicts. So the parametric method makes the portfolio look slightly safer than it really is.

Finally, the backtest recalculated VaR each day using the previous 250 days and checked it against the next day's loss. At 95% confidence, losses should be worse than VaR on about 5% of days. My model had a breach rate of 5.6%, which is close, so the model is broadly reliable but slightly underestimates risk.